# 03 · Modeling — RF vs LR, interval, depreciation

Builds the pipeline, runs `GroupKFold(groups=model_class)`, and produces the Gate evidence on the **pooled** dataset. Logic lives in `ml.train` / `ml.predict`.

In [ ]:
import sys, pathlib
for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_c / 'app').exists() and (_c / 'ml').exists():
        sys.path.insert(0, str(_c)); break
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from ml import ingest, train, metrics
from ml.predict import Predictor
sns.set_theme()
df = ingest.build_engineered_csv(); df.shape

In [ ]:
print('FEATURES:', train.FEATURES)
print('GROUP:', train.GROUP, '| TARGET:', train.TARGET)

In [ ]:
for i, (tr, va) in enumerate(train.iter_group_folds(df, n_splits=5)):
    vc = set(df.iloc[va]['model_class'])
    print(f'fold {i}: train={len(tr)} val={len(va)} val_classes={sorted(vc)}')

### RF vs Ridge baseline (aggregate over GroupKFold folds)
`GroupKFold(groups=model_class)` is the **primary, pessimistic cold-start** view: whole classes are held out, so this scores extrapolation to *unseen* classes. The linear baseline is **Ridge** (plain LinearRegression is numerically unstable here).

In [ ]:
meta = train.main(df=df)  # writes artifacts + returns metrics
rf = meta['models']['random_forest']['aggregate']; rg = meta['models']['ridge']['aggregate']
pd.DataFrame({'RandomForest': {k: v['mean'] for k,v in rf.items()},
              'Ridge': {k: v['mean'] for k,v in rg.items()}})

In [ ]:
print('interval:', meta['interval'])
print('rows by market:', meta['source_markets'])

### Feature importances — is the model using the engineered + enriched signals?

In [ ]:
import joblib
pipe = joblib.load(train.ARTIFACTS_DIR / 'model.joblib')
prep, rf2 = pipe.named_steps['prep'], pipe.named_steps['model']
imp = pd.Series(rf2.feature_importances_, index=prep.get_feature_names_out()).sort_values(ascending=False)
imp.head(20).plot.barh(figsize=(6,6)); plt.gca().invert_yaxis()
plt.title('RF feature importances (top 20)'); plt.tight_layout(); imp.head(20)

### Sanity prediction + per-tree interval
Profile market defaults to `uk` (prediction reads as UK-price-level in RM unless a market is supplied).

In [ ]:
pred = Predictor()
profile = {'model_class':'C Class','year':2018,'age':8,'mileage':90000,
           'transmission':'Automatic','fuel_type':'Petrol','engine_size':2.0,'source_market':'uk'}
print(pred.predict(profile))
trees = pred._per_tree(pred._row(profile))
plt.hist(trees, bins=30)
for pct,c in [(train.INTERVAL_LOW_PCT,'red'),(50,'black'),(train.INTERVAL_HIGH_PCT,'red')]:
    plt.axvline(np.percentile(trees, pct), color=c, ls='--')
plt.title('per-tree predictions'); plt.xlabel('price_rm'); plt.show()

### Depreciation curve

In [ ]:
curve = pd.DataFrame(pred.depreciation(profile, years=6))
sns.lineplot(data=curve, x='year', y='retained_pct', marker='o'); plt.title('retained value'); curve

**Gate:** review the RF-vs-LR table, the empirical interval coverage, feature importances, and these sanity predictions before Phase 03 wires the artifact into the dashboard. Remember the honest caveat: prices are pooled foreign levels in RM, not Malaysian market prices.